# Notebook 07: Campaign Causal Effectiveness & Econometrics
**Focus:** Selection bias proof, Propensity Score Matching (PSM), Difference-in-Differences (DiD), and parallel trends check.


In [ ]:
import duckdb
import pandas as pd
con = duckdb.connect('../data/analytical/retail_bi.duckdb', read_only=True)


### 1. Pre-Treatment Baseline Spend Comparison (Proving Selection Bias)


In [ ]:
pre_spend = con.execute('''
    WITH pre AS (
        SELECT household_key, SUM(sales_value)/32.0 as pre_wk_spend
        FROM fact_transactions WHERE day <= 223 GROUP BY household_key
    )
    SELECT 
        CASE WHEN c.total_campaigns_received > 0 THEN 'Targeted (T=1)' ELSE 'Control (T=0)' END as group_type,
        COUNT(*) as households,
        ROUND(AVG(COALESCE(p.pre_wk_spend, 0)), 2) as avg_pre_weekly_spend
    FROM customer_360 c
    LEFT JOIN pre p ON c.household_key = p.household_key
    GROUP BY 1
''').df()
pre_spend


### 2. Causal Econometric Results Summary


In [ ]:
with open('../reports/causal_campaign_effectiveness.md') as f:
    print(''.join(f.readlines()[15:45]))
